# Notebook 05: Parameter-Efficient Fine-Tuning (PEFT) with LoRA

In this notebook, we use **LoRA (Low-Rank Adaptation)** to fine-tune the Moirai encoder.
Instead of updating millions of parameters, LoRA freezes the foundation model and injects tiny trainable rank-decomposition matrices into the linear layers.

uv pip install peft

In [12]:
import torch
import pandas as pd
from IPython.display import display

from heads import MeanPoolingClassifier, SingleScaleMultiHeadClassifier
from models import LoraHeadWrapper
from utils import get_lsst_dataloaders, universal_grid_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid = [1e-4]
wd_grid = [0.05, 0.01]

BATCH_SIZE = 256

In [13]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [14]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 2. LoRA Baseline: Mean Pooling on Full Sequence (Context + Mask)

In [15]:
methods = ["LoRA", "DoRA", "AdaLoRA"]
df_mean_pool = pd.DataFrame(index=methods, columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"
df_mean_pool.index.name = "PEFT Method"

for method in methods:
    print(f"\n{'='*40}\nTraining with {method}\n{'='*40}")
    for patch in PATCH_SIZES:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")

        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 8,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_mean_pool.loc[method, patch] = metrics["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"MeanPool-{method}"] = metrics

print("Mean Pooling (LoRA/DoRA/AdaLoRA) - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))


Training with LoRA
LR=0.0001| WD=0.05
 Early stopping : 39
Val Loss: 1.0365
LR=0.0001| WD=0.01
 Early stopping : 30
Val Loss: 1.0844
LR=0.0001| WD=0.05
 Early stopping : 36
Val Loss: 1.0698
LR=0.0001| WD=0.01
 Early stopping : 34
Val Loss: 1.0885
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.0802
LR=0.0001| WD=0.01
 Early stopping : 35
Val Loss: 1.0709
LR=0.0001| WD=0.05
 Early stopping : 30
Val Loss: 1.1617
LR=0.0001| WD=0.01
 Early stopping : 32
Val Loss: 1.1191

Training with DoRA
LR=0.0001| WD=0.05


KeyboardInterrupt: 

## 3. LoRA Rank Ablation Study
Testing the impact of the bottleneck rank $r$ on the MHA performance. A higher rank means more trainable parameters.

In [ ]:
PATCH = 8
RANKS_TO_TEST = [2, 4, 8, 16, 32, 64]
methods = ["LoRA", "DoRA", "AdaLoRA"]

df_rank_ablation8 = pd.DataFrame(index=methods, columns=RANKS_TO_TEST)
df_rank_ablation8.index.name = "PEFT Method"
df_rank_ablation8.columns.name = f"Rank 'r' (Patch {PATCH})"

for method in methods:
    for r in RANKS_TO_TEST:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")

        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": r,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            lr_grid=lr_grid, wd_grid=wd_grid
        )
        df_rank_ablation8.loc[method, r] = metrics["Accuracy"]
        df_patch_8_metrics.loc[f"{method} r={r}"] = metrics

print(f"Rank Ablation (Patch = {PATCH}) - Accuracy")
display(df_rank_ablation8.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=0.0001| WD=0.05
 Early stopping : 54
Val Loss: 1.0520
LR=0.0001| WD=0.05
 Early stopping : 44
Val Loss: 1.0809
LR=0.0001| WD=0.05
 Early stopping : 41
Val Loss: 1.0879
LR=0.0001| WD=0.05
 Early stopping : 34
Val Loss: 1.0588
LR=0.0001| WD=0.05
 Early stopping : 31
Val Loss: 1.0559
LR=0.0001| WD=0.05
 Early stopping : 22
Val Loss: 1.0690
LR=0.0001| WD=0.05
 Early stopping : 56
Val Loss: 1.0553
LR=0.0001| WD=0.05
 Early stopping : 46
Val Loss: 1.0733
LR=0.0001| WD=0.05
 Early stopping : 37
Val Loss: 1.0871
LR=0.0001| WD=0.05
 Early stopping : 26
Val Loss: 1.0953
LR=0.0001| WD=0.05
 Early stopping : 28
Val Loss: 1.0537
LR=0.0001| WD=0.05
 Early stopping : 20
Val Loss: 1.0932
LR=0.0001| WD=0.05
 Early stopping : 107
Val Loss: 1.0940
LR=0.0001| WD=0.05
 Early stopping : 112
Val Loss: 1.0613
LR=0.0001| WD=0.05
 Early stopping : 91
Val Loss: 1.0688
LR=0.0001| WD=0.05
 Early stopping : 83
Val Loss: 1.0738
LR=0.0001| WD=0.05
 Early stopping : 67
Val Loss: 1.0672
LR=0.0001| WD=0.05
 Early sto

Rank 'r' (Patch 8),2,4,8,16,32,64
PEFT Method,,,,,,
LoRA,0.6419,0.6448,0.6480,0.6423,0.6500,0.6472
DoRA,0.6346,0.6294,0.6338,0.6371,0.6419,0.6302
AdaLoRA,0.6290,0.6342,0.6318,0.6294,0.6371,0.6330



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6436,0.5026,0.3789,0.3851,0.5975,0.6436,0.5905
MeanPool-DoRA,0.6488,0.4401,0.3893,0.3976,0.5896,0.6488,0.6033
MeanPool-AdaLoRA,0.6403,0.4374,0.3922,0.3997,0.5879,0.6403,0.6025
LoRA r=2,0.6419,0.4871,0.3905,0.3985,0.5997,0.6419,0.6008
LoRA r=4,0.6448,0.4522,0.3941,0.3963,0.5885,0.6448,0.6039
LoRA r=8,0.6480,0.4884,0.3691,0.3777,0.6014,0.6480,0.5973
LoRA r=16,0.6423,0.4492,0.3782,0.3846,0.5777,0.6423,0.5859
LoRA r=32,0.6500,0.4464,0.3803,0.3828,0.5880,0.6500,0.6004
LoRA r=64,0.6472,0.4761,0.3997,0.4009,0.5999,0.6472,0.6110


## 4. Advanced LoRA: Multi-Head Attention

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = [f"MHA (Heads={NUM_HEADS} | LoRA r=64)"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_mha = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 64
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = metrics_mha["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"MHA-16 ({mode}) LoRA r=64"] = metrics_mha

LR=0.0001| WD=0.05
 Early stopping : 17
Val Loss: 1.0205
LR=0.0001| WD=0.05
 Early stopping : 18
Val Loss: 1.1003
LR=0.0001| WD=0.05
 Early stopping : 15
Val Loss: 1.1124
LR=0.0001| WD=0.05
 Early stopping : 16
Val Loss: 1.0355
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.1456
LR=0.0001| WD=0.05
 Early stopping : 15
Val Loss: 1.1193
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.1238
LR=0.0001| WD=0.05
 Early stopping : 14
Val Loss: 1.1710


In [ ]:
print("MHA with LoRA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

MHA with LoRA - Accuracy


Patch Size                                          8       16      32      64
Model                      Mode                                               
MHA (Heads=16 | LoRA r=64) shared_context       0.6225  0.6034  0.6148  0.6208
                           independent_context  0.6346  0.6496  0.6062  0.6221


Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6436,0.5026,0.3789,0.3851,0.5975,0.6436,0.5905
MeanPool-DoRA,0.6488,0.4401,0.3893,0.3976,0.5896,0.6488,0.6033
MeanPool-AdaLoRA,0.6403,0.4374,0.3922,0.3997,0.5879,0.6403,0.6025
LoRA r=2,0.6419,0.4871,0.3905,0.3985,0.5997,0.6419,0.6008
LoRA r=4,0.6448,0.4522,0.3941,0.3963,0.5885,0.6448,0.6039
LoRA r=8,0.6480,0.4884,0.3691,0.3777,0.6014,0.6480,0.5973
LoRA r=16,0.6423,0.4492,0.3782,0.3846,0.5777,0.6423,0.5859
LoRA r=32,0.6500,0.4464,0.3803,0.3828,0.5880,0.6500,0.6004
LoRA r=64,0.6472,0.4761,0.3997,0.4009,0.5999,0.6472,0.6110
